In [22]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.stats.multitest import multipletests

# Set this to the location of your downloaded SoLo dataset
DATASET_DIR = Path("path/to/SoLo_dataset")

# Mobility cross-modal correlations

In [23]:
label_cols = [  "feel_lonely", "feel_isolated", "feel_connected"]

all_daily_ema = []

for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("participant_"):
        continue

    ema_path = os.path.join( DATASET_DIR, participant,  "Self_Report",  "ema_daily.csv")

    if not os.path.exists(ema_path):
        print(f"No EMA file: {participant}")
        continue

    df = pd.read_csv(ema_path)

    # Unix timestamp (ms) -> America/Los_Angeles local time
    df["datetime_la"] = (pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles"))

    # Local calendar date
    df["date"] = df["datetime_la"].dt.date

    # Daily mean of the three EMA labels
    daily = (df.groupby("date", as_index=False)[label_cols].mean())

    daily.insert(0, "participant", participant)

    all_daily_ema.append(daily)

daily_ema = pd.concat(all_daily_ema, ignore_index=True)

print("Participants:", daily_ema["participant"].nunique())
print("Daily rows:", len(daily_ema))

No EMA file: participant_groups.csv
Participants: 31
Daily rows: 3777


In [24]:
all_mobility = []
for participant in sorted(os.listdir(DATASET_DIR)):

    if not participant.startswith("participant_"):
        continue

    mobility_path = os.path.join( DATASET_DIR, participant, "AWARE","mobility_features.csv")

    if not os.path.exists(mobility_path):
        print(f"No mobility file: {participant}")
        continue

    df = pd.read_csv(mobility_path)

    df["date"] = pd.to_datetime(df["date"]).dt.date

    all_mobility.append(df)

daily_mobility = pd.concat(all_mobility, ignore_index=True)

print("Participants:", daily_mobility["participant"].nunique())
print("Daily mobility rows:", len(daily_mobility))

No mobility file: participant_18
No mobility file: participant_27
No mobility file: participant_groups.csv
Participants: 29
Daily mobility rows: 2740


In [25]:
merged = daily_mobility.merge(daily_ema, on=["participant", "date"], how="inner")
display(merged)

,participant,date,locationvariance,loglocationvariance,totaldistance,avgspeed,varspeed,numberofsignificantplaces,numberlocationtransitions,radiusgyration,...,avglengthstayatclusters,stdlengthstayatclusters,locationentropy,normalizedlocationentropy,timeathome,homelabel,minutesdataused,feel_lonely,feel_isolated,feel_connected
0,participant_1,2022-01-30,3.581604e-07,-6.445922,250.751521,1.827069,0.069280,1.0,0.0,0.0,...,89.391800,0.0,0.0,0.0,89.391800,1.0,97.626350,0.666667,1.333333,32.000000
1,participant_1,2022-02-02,4.284296e-06,-5.368121,1092.894346,4.464509,1.436016,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,14.687767,0.000000,3.000000,97.000000
2,participant_1,2022-02-11,2.077151e-28,-27.682532,16.534166,1.652499,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.600333,0.000000,0.000000,100.000000
3,participant_1,2022-02-12,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.0,...,19.506433,0.0,0.0,0.0,19.506433,1.0,19.506433,24.333333,20.666667,0.000000
4,participant_1,2022-02-16,0.000000e+00,NaN,30.642937,12.342199,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.148967,0.000000,1.000000,50.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1857,participant_9,2022-06-06,2.030151e-28,-27.692472,0.000000,0.000000,0.000000,1.0,0.0,0.0,...,3.172100,0.0,0.0,0.0,0.000000,1.0,3.172100,22.000000,7.666667,66.666667
1858,participant_9,2022-06-07,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.0,...,5.311283,0.0,0.0,0.0,0.000000,1.0,5.311283,15.666667,5.333333,74.000000
1859,participant_9,2022-06-09,0.000000e+00,NaN,105.378940,2.181621,0.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.898183,6.000000,0.500000,89.000000
1860,participant_9,2022-06-15,0.000000e+00,NaN,0.000000,0.000000,0.000000,1.0,0.0,0.0,...,2.016833,0.0,0.0,0.0,0.000000,1.0,2.016833,13.000000,3.000000,98.000000


In [26]:
mobility_features = ["locationvariance", "totaldistance", "varspeed", "radiusgyration", "timeattop1location", "avglengthstayatclusters"]

participant_corrs = []
for feature in mobility_features:
    for label in label_cols:
        for participant, df_p in merged.groupby("participant"):
            pair = df_p[[feature, label]].dropna()
            if len(pair) < 3 or pair[feature].nunique() < 2 or pair[label].nunique() < 2:
                continue
            r, _ = pearsonr(pair[feature], pair[label])
            z = np.arctanh(np.clip(r, -0.999999, 0.999999))
            participant_corrs.append({"participant": participant, "mobility_feature": feature, "ema_label": label, "n_days": len(pair), "r": r, "z": z})

participant_corrs = pd.DataFrame(participant_corrs)

group_results = []

for (feature, label), df_pair in participant_corrs.groupby(["mobility_feature", "ema_label"]):
    z_values = df_pair["z"].dropna().values
    if len(z_values) < 2:
        continue
    mean_z = np.mean(z_values)
    mean_r = np.tanh(mean_z)
    t_stat, p_raw = ttest_1samp(z_values, popmean=0, alternative="two-sided")
    group_results.append({"mobility_feature": feature, "ema_label": label, "n_participants": len(z_values), "mean_r": mean_r, "mean_z": mean_z, "t_stat": t_stat, "p_raw": p_raw})

mobility_results = pd.DataFrame(group_results)
mobility_results["p_fdr"] = multipletests(mobility_results["p_raw"].values, alpha=0.05, method="fdr_bh")[1]
mobility_results["sig_fdr"] = mobility_results["p_fdr"].apply(lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")
mobility_results["table_value"] = mobility_results.apply(lambda row: f"{row['mean_r']:.2f}{row['sig_fdr']} (N={int(row['n_participants'])})", axis=1)

final_table = mobility_results.pivot(index="mobility_feature", columns="ema_label", values="table_value")
final_table = final_table.reindex(mobility_features)
final_table = final_table[["feel_lonely", "feel_isolated", "feel_connected"]]

final_table

ema_label,feel_lonely,feel_isolated,feel_connected
mobility_feature,,,
locationvariance,-0.10** (N=28),-0.08* (N=29),0.07 (N=29)
totaldistance,-0.08* (N=28),-0.06 (N=29),0.10* (N=29)
varspeed,-0.12*** (N=28),-0.09* (N=29),0.10* (N=29)
radiusgyration,-0.09* (N=27),-0.07* (N=28),0.08 (N=28)
timeattop1location,0.07 (N=27),0.14 (N=28),-0.11* (N=28)
avglengthstayatclusters,0.09* (N=27),0.18 (N=28),-0.14** (N=28)


# Oura sleep cross-modal correlations

In [28]:
all_morning_ema = []

for participant in sorted(os.listdir(DATASET_DIR)):
    if not participant.startswith("participant_"):
        continue

    ema_path = DATASET_DIR / participant / "Self_Report" / "ema_daily.csv"
    if not ema_path.exists():
        continue

    df = pd.read_csv(ema_path)

    df = df[df["surveyindex"] == 0].copy()

    # Convert EMA timestamps to Pacific time
    df["datetime_la"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
    df["date"] = df["datetime_la"].dt.date
    df["ema_sleep_la"] = pd.to_datetime(df["sleep_time"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
    df["ema_wake_la"] = pd.to_datetime(df["wake_time"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")

    # Calculate self-reported sleep duration
    df["ema_sleep_duration_hours"] = (df["ema_wake_la"] - df["ema_sleep_la"]).dt.total_seconds() / 3600

    all_morning_ema.append(df)

morning_ema = pd.concat(all_morning_ema, ignore_index=True)

print("Participants:", morning_ema["participant"].nunique())
print("Morning EMA records:", len(morning_ema))

Participants: 31
Morning EMA records: 3306


In [29]:
all_oura = []

for participant in sorted(os.listdir(DATASET_DIR)):
    if not participant.startswith("participant_"):
        continue

    oura_path = DATASET_DIR / participant / "Oura" / "sleep_daily.csv"
    if not oura_path.exists():
        continue

    df = pd.read_csv(oura_path)

    # Convert Oura timestamps to Pacific time
    df["oura_bedtime_la"] = pd.to_datetime(df["sleep_bedtime_start"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
    df["oura_wake_la"] = pd.to_datetime(df["sleep_bedtime_end"], unit="ms", utc=True).dt.tz_convert("America/Los_Angeles")
    df["wake_date"] = df["oura_wake_la"].dt.date

    # Convert Oura sleep duration from seconds to hours
    df["oura_sleep_duration_hours"] = df["sleep_duration"] / 3600

    all_oura.append(df)

daily_oura = pd.concat(all_oura, ignore_index=True)

print("Participants:", daily_oura["participant"].nunique())
print("Oura sleep records:", len(daily_oura))

Participants: 31
Oura sleep records: 4283


In [30]:
# Keep longest Oura sleep session per participant-date
daily_oura = daily_oura.sort_values("sleep_duration", ascending=False).drop_duplicates(["participant", "wake_date"])

# Keep earliest morning EMA submission per participant-date
morning_ema = morning_ema.sort_values("timestamp").drop_duplicates(["participant", "date"])

# Merge Oura with morning EMA
merged_sleep = daily_oura.merge(morning_ema, left_on=["participant", "wake_date"], right_on=["participant", "date"], how="inner", suffixes=("_oura", "_ema"), validate="one_to_one")

print("Matched records:", len(merged_sleep))
print("Participants:", merged_sleep["participant"].nunique())

# Display matched records with source-specific labels
display(merged_sleep[["participant", "wake_date", "sleep_score", "quality_score", "rested_score", "oura_sleep_duration_hours", "ema_sleep_duration_hours"]].head(10).rename(columns={
    "sleep_score": "oura_sleep_score",
    "quality_score": "ema_quality_score",
    "rested_score": "ema_rested_score"
}))

Matched records: 2792
Participants: 31


,participant,wake_date,oura_sleep_score,ema_quality_score,ema_rested_score,oura_sleep_duration_hours,ema_sleep_duration_hours
0,participant_11,2022-04-08,83.0,52.0,74.0,15.000000,-14.615278
1,participant_17,2022-02-05,81.0,39.0,38.0,15.000000,9.582500
2,participant_12,2022-02-14,68.0,64.0,66.0,15.000000,-14.996389
3,participant_15,2022-05-15,63.0,31.0,40.0,15.000000,5.238333
4,participant_15,2022-03-01,78.0,27.0,67.0,15.000000,12.970278
5,participant_15,2022-01-25,51.0,19.0,20.0,15.000000,4.422222
6,participant_15,2022-04-05,78.0,36.0,79.0,15.000000,-14.879444
7,participant_21,2022-03-18,78.0,50.0,41.0,14.766667,7.385556
8,participant_15,2022-05-03,72.0,45.0,61.0,14.716667,7.005833
9,participant_17,2022-05-07,81.0,67.0,64.0,14.583333,8.302500


In [31]:
sleep_pairs = [ ("sleep_score", "quality_score"),
                ("sleep_score", "rested_score"),
                ("oura_sleep_duration_hours", "ema_sleep_duration_hours")]

participant_sleep_corrs = []

for oura_var, ema_var in sleep_pairs:
    for participant, df_p in merged_sleep.groupby("participant"):
        pair = df_p[[oura_var, ema_var]].dropna()

        # Keep plausible self-reported sleep durations
        if ema_var == "ema_sleep_duration_hours":
            pair = pair[pair[ema_var].between(3, 16)]

        if len(pair) < 3 or pair[oura_var].nunique() < 2 or pair[ema_var].nunique() < 2:
            continue

        r, _ = pearsonr(pair[oura_var], pair[ema_var])
        z = np.arctanh(np.clip(r, -0.999999, 0.999999))

        participant_sleep_corrs.append({
            "participant": participant,
            "oura_var": oura_var,
            "ema_var": ema_var,
            "n_days": len(pair),
            "r": r,
            "z": z
        })

participant_sleep_corrs = pd.DataFrame(participant_sleep_corrs)

# Aggregate participant correlations using Fisher's z
group_results = []

for (oura_var, ema_var), df_pair in participant_sleep_corrs.groupby(["oura_var", "ema_var"]):
    z_values = df_pair["z"].dropna().values

    if len(z_values) < 2:
        continue

    mean_z = np.mean(z_values)
    mean_r = np.tanh(mean_z)
    t_stat, p_raw = ttest_1samp(z_values, popmean=0, alternative="two-sided")

    group_results.append({
        "oura_var": oura_var,
        "ema_var": ema_var,
        "n_participants": len(z_values),
        "mean_r": mean_r,
        "mean_z": mean_z,
        "t_stat": t_stat,
        "p_raw": p_raw
    })

final_sleep_results = pd.DataFrame(group_results)

# Apply BH-FDR correction across all three tests
final_sleep_results["p_fdr"] = multipletests(final_sleep_results["p_raw"], method="fdr_bh")[1]

final_sleep_results["oura_var"] = final_sleep_results["oura_var"].replace({"sleep_score": "oura_sleep_score"})
final_sleep_results["ema_var"] = final_sleep_results["ema_var"].replace({"quality_score": "ema_quality_score", "rested_score": "ema_rested_score"})

display(final_sleep_results)

,oura_var,ema_var,n_participants,mean_r,mean_z,t_stat,p_raw,p_fdr
0,oura_sleep_duration_hours,ema_sleep_duration_hours,29,0.811552,1.131557,14.831760,8.650236e-15,2.595071e-14
1,oura_sleep_score,ema_quality_score,31,0.233766,0.238170,3.397509,1.936242e-03,1.936242e-03
2,oura_sleep_score,ema_rested_score,31,0.314919,0.325996,6.064731,1.163665e-06,1.745497e-06
